# Tutorial 1 using Unit Commitment Example

This tutorial shows the different commands for using GEMSPy python package for modeling and simulating an Unit Commitment problem.

We use in this tutorial the library, *antares-legacy-models*, present in GEMS repo.

## Installation of GEMSPy

In [ ]:
pip install gemspy

: 

## Create a simple system composed of an area with load

### Step 1: Configure an Area

An Area is the central node in the electrical grid where different components connect. We'll create an area with spillage cost and unsupplied energy cost parameters.

In [ ]:
from gems.study.parsing import InputComponent, InputComponentParameter, InputSystem
from gems.model.parsing import parse_yaml_library
from gems.model.resolve_library import resolve_library
from pathlib import Path

# Set tutorial directory structure
study_name = "QSE_2_Unit_Commitment_tuto"

base_path = Path.cwd()
library_folder = base_path / study_name / "input" / "model-libraries"
library_file = next(library_folder.glob("*.yml")).name

series_folder = base_path / study_name / "input" / "data-series"

print("LIBRARY READING")
print("\tLibrary path:", library_file)

with open(Path(library_folder, library_file)) as lib_file:
    input_libraries = [parse_yaml_library(lib_file)]
    library_name = input_libraries[0].id
    print("\tLibrary loaded:", library_name)

result_lib = resolve_library(input_libraries)

print("COMPONENTS DEFINITION")
components = []

# Add an area component with spillage and ENS costs
components.append(
    InputComponent(
        id="my_area",
        model=f"{library_name}.area",
        parameters=[
            InputComponentParameter(
                id="spillage_cost",
                time_dependent=False,
                scenario_dependent=False,
                value=1000),
            InputComponentParameter(
                id="ens_cost",
                time_dependent=False,
                scenario_dependent=False,
                value=10000),
        ],
    )
)

components.append(
    InputComponent(
        id="load",
        model=f"{library_name}.load",
        parameters=[
            InputComponentParameter(
                id="load",
                time_dependent=True,
                scenario_dependent=True,
                value="load"),
        ],
    )
)
print("\tComponents defined:", [comp.id for comp in components])

print("INPUT SYSTEM CREATION")
input_system = InputSystem(
            id = "my_system",
            model_libraries=library_name,
            components=components,
        )
print("\tThe following input system has been created :")
print(input_system)

### Step 2: Configure the Generator (1 unit of 100 MW)

Now we add a thermal generator to the area. This generator will have:
- 1 unit with a maximum capacity of 100 MW
- Generation cost of 50 $/MWh

In [ ]:
components.append(
    InputComponent(
        id="thermal_gen",
        model=f"{library_name}.thermal",
        parameters=[
            InputComponentParameter(id="p_min_unit", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="p_max_unit", time_dependent=False, scenario_dependent=False, value=100),
            InputComponentParameter(id="generation_cost", time_dependent=False, scenario_dependent=False, value=50),
            InputComponentParameter(id="startup_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="fixed_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="d_min_up", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="d_min_down", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="p_min_cluster", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="p_max_cluster", time_dependent=False, scenario_dependent=False, value=100),
            InputComponentParameter(id="nb_units_max", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="nb_units_min", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_forward", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_backward", time_dependent=False, scenario_dependent=False, value=0)
        ],
    )
)

print("\tComponents defined:", [comp.id for comp in components])

print("INPUT SYSTEM CREATION")
input_system = InputSystem(
            id = "my_system",
            model_libraries=library_name,
            components=components,
        )
print("\tThe following input system has been created :")
print(input_system)

## Step 3 : Configure the port connection

Then, we interconnect the two components by configuring the ports connection

In [ ]:
from gems.study.parsing import InputPortConnections

connections = []

connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="thermal_gen",
        port2="balance_port",
    )
)

connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="load",
        port2="balance_port",
    )
)


print("\tConnections defined:")
for conn in connections:
    print(f"\t\t{conn.component1}:{conn.port1} <-> {conn.component2}:{conn.port2}")

input_system = InputSystem(
            components=components,
            connections=connections,
        )
print("\tThe following input system has been created :")
print(input_system)



## Step 4 : Run the study and Graph results

In [ ]:
from gems.simulation.optimization import (
    TimeBlock,
    build_problem,
)
from gems.study.resolve_components import (
    build_data_base,
    build_network,
    resolve_system
)
from gems.simulation.output_values import OutputValues

ScenarioNumber = 1
TimeSpan = 7*24
result_lib = resolve_library(input_libraries)
components_input = resolve_system(input_system, result_lib)
database = build_data_base(input_system, Path(series_folder))

network = build_network(components_input)
problem = build_problem(
    network,
    database,
    TimeBlock(1, [i for i in range(0, TimeSpan)]),
    ScenarioNumber,
)


try:
    status = problem.solver.Solve()
    objective_value = problem.solver.Objective().Value()
    print("✓ Optimization problem solved successfully.")
    results = OutputValues(problem)
    print(f"✓ Results extracted successfully.")
except Exception as e:
    print(f"Error solving optimization problem: {e}")
    raise e


# PRINTING RESULTS

print("\n","="*15," RESULTS ","="*15)
print(f"\tSolver status: {status}")
objective_value = results.problem.solver.Objective().Value()
print(f"\tObjective value: {objective_value}")
print(f"\nComponent variables and outputs:")
print(results)

# Interconnect Wind Farm and Solar Farm

In this part, a wind farm and a solar are interconnected to our system

## Step 1 : Creation of componnents 

In [ ]:


components.append(
    InputComponent(
        id="wind_farm",
        model=f"{library_name}.renewable",
        parameters=[
            InputComponentParameter(id="nominal_capacity", time_dependent=False, scenario_dependent=False, value=60),
            InputComponentParameter(id="unit_count", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="generation", time_dependent=True, scenario_dependent=True, value="wind"),
        ],              
    )
)

connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="wind_farm",
        port2="balance_port",
    )
)

components.append(
    InputComponent(
        id="solar_farm",
        model=f"{library_name}.renewable",
        parameters=[
            InputComponentParameter(id="nominal_capacity", time_dependent=False, scenario_dependent=False, value=40),
            InputComponentParameter(id="unit_count", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="generation", time_dependent=True, scenario_dependent=True, value="solar"),
        ],              
    )
)

connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="solar_farm",
        port2="balance_port",
    )
)

print("\tComponents defined:", [comp.id for comp in components])
print("\tConnections defined:")
for conn in connections:
    print(f"\t\t{conn.component1}:{conn.port1} <-> {conn.component2}:{conn.port2}")

input_system = InputSystem(
    id = "my_system",
    components=components,
    connections=connections,
    )

print("\tThe following input system has been created :")
print(input_system)

## Step 2 : Results visualization

In [ ]:
result_lib = resolve_library(input_libraries)
components_input = resolve_system(input_system, result_lib)
database = build_data_base(input_system, Path(series_folder))

network = build_network(components_input)
problem = build_problem(
    network,
    database,
    TimeBlock(1, [i for i in range(0, TimeSpan)]),
    ScenarioNumber,
)


try:
    status = problem.solver.Solve()
    objective_value = problem.solver.Objective().Value()
    print("✓ Optimization problem solved successfully.")
    results = OutputValues(problem)
    print(f"✓ Results extracted successfully.")
except Exception as e:
    print(f"Error solving optimization problem: {e}")
    raise e


print("\n","="*15," RESULTS ","="*15)
print(f"\tSolver status: {status}")
objective_value = results.problem.solver.Objective().Value()
print(f"\tObjective value: {objective_value}")
print(f"\nComponent variables and outputs:")
print(results)

# Add Unit Commitment

Configure the number of units of thermal generator

## Add Units in the thermal cluster

In [ ]:
components = [comp for comp in components if comp.id != 'thermal_gen']

components.append(
    InputComponent(
        id="thermal_gen",
        model=f"{library_name}.thermal",
        parameters=[
            InputComponentParameter(id="p_min_unit", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="p_max_unit", time_dependent=False, scenario_dependent=False, value=10),
            InputComponentParameter(id="generation_cost", time_dependent=False, scenario_dependent=False, value=50),
            InputComponentParameter(id="startup_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="fixed_cost", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="d_min_up", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="d_min_down", time_dependent=False, scenario_dependent=False, value=1),
            InputComponentParameter(id="p_min_cluster", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="p_max_cluster", time_dependent=False, scenario_dependent=False, value=100),
            InputComponentParameter(id="nb_units_max", time_dependent=False, scenario_dependent=False, value=10),
            InputComponentParameter(id="nb_units_min", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_forward", time_dependent=False, scenario_dependent=False, value=0),
            InputComponentParameter(id="nb_units_max_variation_backward", time_dependent=False, scenario_dependent=False, value=0)
        ],
    )
)

connections.append(
    InputPortConnections(
        component1="my_area",
        port1="balance_port",
        component2="thermal_gen",
        port2="balance_port",
    )
)   

print("\tComponents defined:", [comp.id for comp in components])
print("\tConnections defined:")
for conn in connections:
    print(f"\t\t{conn.component1}:{conn.port1} <-> {conn.component2}:{conn.port2}")

input_system = InputSystem(
    id = "my_system",
    components=components,
    connections=connections,
    )   

## Results

In [ ]:
result_lib = resolve_library(input_libraries)
components_input = resolve_system(input_system, result_lib)
database = build_data_base(input_system, Path(series_folder))

network = build_network(components_input)
problem = build_problem(
    network,
    database,
    TimeBlock(1, [i for i in range(0, TimeSpan)]),
    ScenarioNumber,
)


try:
    status = problem.solver.Solve()
    objective_value = problem.solver.Objective().Value()
    print("✓ Optimization problem solved successfully.")
    results = OutputValues(problem)
    print(f"✓ Results extracted successfully.")
except Exception as e:
    print(f"Error solving optimization problem: {e}")
    raise e


print("\n","="*15," RESULTS ","="*15)
print(f"\tSolver status: {status}")
objective_value = results.problem.solver.Objective().Value()
print(f"\tObjective value: {objective_value}")
print(f"\nComponent variables and outputs:")
print(results)

In [ ]:
print(results.problem.solver)